# 02 · BEV 特征融合与鲁棒性

本 notebook 不调用真实 nuScenes 数据，而是用可控的 synthetic BEV feature map 研究融合接口。这样可以把一个常被论文名词遮住的问题拆开：当某个模态掉线、错位或噪声增大时，融合模块如何退化？

设 camera 与 LiDAR 在 BEV 网格上的特征为 (f_c,f_l)，最简单的融合是：

\[
f_{fuse}=\alpha f_c+(1-\alpha)f_l
\]

真实系统会学习更复杂的 attention/gating，但首先要理解这个接口的误差传播。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

rng = np.random.default_rng(11)
H, W = 96, 128
yy, xx = np.mgrid[0:H, 0:W]

def gaussian(cx, cy, sx, sy, amplitude=1.0):
    return amplitude * np.exp(-(((xx-cx)**2)/(2*sx**2) + ((yy-cy)**2)/(2*sy**2)))

target = (gaussian(35, 48, 5, 3) + gaussian(76, 35, 7, 4) + gaussian(94, 73, 4, 6))
target = target / target.max()

camera = np.clip(target + .28*rng.normal(size=(H, W)), 0, None)
lidar = np.clip(.85*target + .10*rng.normal(size=(H, W)), 0, None)
camera[:, :8] *= .25  # camera blind/low-quality strip
lidar[55:70, 52:70] *= .35  # sparse LiDAR region

def normalize(x):
    x = x - x.min()
    return x / (x.max() + 1e-8)

target, normalize(camera).shape

In [ ]:
def fuse(camera_feature, lidar_feature, camera_weight=.5):
    c = normalize(camera_feature)
    l = normalize(lidar_feature)
    return camera_weight*c + (1-camera_weight)*l

def iou(pred, truth, threshold=.45):
    p, t = pred > threshold, truth > threshold
    return (p & t).sum() / ((p | t).sum() + 1e-8)

def show_fusion(camera_weight=.5):
    fused = fuse(camera, lidar, camera_weight)
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
    for ax, image, title in zip(axes, [target, normalize(camera), normalize(lidar), fused], ['target', 'camera', 'LiDAR', f'fused α={camera_weight:.2f}']):
        im = ax.imshow(image, cmap='magma', vmin=0, vmax=1)
        ax.set_title(title); ax.set_axis_off()
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=.7)
    fig.suptitle(f'IoU = {iou(fused, target):.3f}', y=1.02)
    plt.show()

show_fusion(.5)

In [ ]:
interact(show_fusion, camera_weight=FloatSlider(min=0, max=1, step=.05, value=.5, description='camera α'));

## 1. Modality dropout 与时间错位

对融合系统而言，“模态存在”不等于“模态可靠”。下面比较正常、camera dropout、LiDAR dropout 和空间错位四种状态。

**TODO**：先预测哪一种状态的 IoU 会下降最多，再运行代码。然后解释为什么全局平均 IoU 可能掩盖一个关键目标的完全丢失。

In [ ]:
def evaluate_conditions():
    conditions = {
        'normal': (camera, lidar),
        'camera_dropout': (np.zeros_like(camera), lidar),
        'lidar_dropout': (camera, np.zeros_like(lidar)),
        'spatial_misalignment': (np.roll(camera, 5, axis=1), lidar),
    }
    rows = []
    for name, (c, l) in conditions.items():
        fused = fuse(c, l, .5)
        rows.append((name, iou(fused, target), float(np.mean(np.abs(fused-target)))))
    return rows

for row in evaluate_conditions():
    print(f'{row[0]:24s} IoU={row[1]:.3f}  MAE={row[2]:.3f}')

## 2. 质量感知融合

固定 (alpha) 是教学基线。真实模型可能根据 sensor health、天气、遮挡和 feature uncertainty 产生空间变化的 gate：

\[
f_{fuse}(u,v)=g(u,v)f_c(u,v)+(1-g(u,v))f_l(u,v)
\]

下面使用一个非常简化的质量估计：camera 的局部方差越大，camera 权重越低；LiDAR 的局部强度越低，LiDAR 权重越低。它不是生产算法，只用于展示“质量估计 → 融合权重”的接口。

In [ ]:
def local_mean(x, radius=2):
    padded = np.pad(x, radius, mode='edge')
    acc = np.zeros_like(x, dtype=float)
    for dy in range(2*radius+1):
        for dx in range(2*radius+1):
            acc += padded[dy:dy+H, dx:dx+W]
    return acc / (2*radius+1)**2

def quality_aware_fuse(camera_feature, lidar_feature):
    c, l = normalize(camera_feature), normalize(lidar_feature)
    c_noise = np.abs(c - local_mean(c))
    c_quality = 1 / (1 + 3*c_noise)
    l_quality = np.clip(l + .15, 0, 1)
    gate = c_quality / (c_quality + l_quality + 1e-8)
    fused = gate*c + (1-gate)*l
    return fused, gate

fused_adaptive, gate = quality_aware_fuse(camera, lidar)
print('fixed IoU   :', round(iou(fuse(camera, lidar, .5), target), 3))
print('adaptive IoU:', round(iou(fused_adaptive, target), 3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, image, title in zip(axes, [gate, fused_adaptive, target], ['learned-like gate', 'adaptive fusion', 'target']):
    ax.imshow(image, cmap='magma', vmin=0, vmax=1); ax.set_title(title); ax.set_axis_off()
plt.tight_layout()

## 完成标准

- 能区分 fixed-weight fusion 与 quality-aware gating；
- 能说明 modality dropout、标定误差和时间错位不是同一种 failure；
- 设计一个自己的 gate，并比较固定权重、你的 gate 和单模态 baseline；
- 报告平均指标之外的 per-region/per-object failure，说明为什么某种退化对安全更重要。